In [1]:
import pandas as pd
from pandas import col

df = pd.read_parquet('../data/raw/cps_filers_clean.parquet')

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Shape: (64696, 59)

Column names:
['year', 'serial', 'month', 'cpsid', 'asecflag', 'asecwth', 'statefip', 'hhincome', 'pernum', 'cpsidp', 'cpsidv', 'asecwt', 'age', 'sex', 'race', 'marst', 'vetstat', 'famsize', 'nchild', 'nchlt5', 'ftype', 'yrimmig', 'citizen', 'nativity', 'hispan', 'empstat', 'occ2010', 'ind', 'classwkr', 'educ', 'diffany', 'wkswork1', 'uhrsworkly', 'pension', 'firmsize', 'ftotval', 'inctot', 'incwage', 'incbus', 'incss', 'incretir', 'incint', 'incdivid', 'incrent', 'ctccrd', 'actccrd', 'adjginc', 'eitcred', 'fedtaxac', 'fica', 'filestat', 'depstat', 'margtax', 'taxinc', 'spmwt', 'spmfedtaxac', 'spmftotval', 'spmfamunit', 'eff_rate']


In [5]:
# Step 1: Replace uhrsworkly 999 with (or 0) & Replace occ2010 9999 with blank or "not working".
#999 in uhrsworkly means "did not work" not actual hours.
df['uhrsworkly'] = df['uhrsworkly'].replace(999,pd.NA)
df['occ2010'] = df['occ2010'].replace(9999,pd.NA)

print("uhrsworkly 999 count:",(df['uhrsworkly'] == 999).sum())
print("occ2010 9999 count:", (df['occ2010'] == 9999).sum())

uhrsworkly 999 count: 0
occ2010 9999 count: 0


In [10]:
# Step 2: Fill incretir with 0 for every person under 58 years of age.
# File partially cleaned.
df.loc[df['age'] < 58, 'incretir'] = 0
# incretir shows 0 NaN values
print("incretir NaN count:", (df['incretir'] == 0).sum())

incretir NaN count: 61485


In [12]:
# Step 3: Handle the "0 = not asked" codes in pension, firmsize, vetstat, diffany, and nativity.
df['pension'] = df['pension'].replace(0, pd.NA)
df['firmsize'] = df['firmsize'].replace(0, pd.NA)
df['vetstat'] = df['vetstat'].replace(0,pd.NA)
df['diffany'] = df['diffany'].replace(0,pd.NA)
df['nativity'] = df['nativity'].replace(0, pd.NA)
 # Check all counts to 0
print("pension 0 count:", (df['pension'] == 0).sum())
print("firmsize 0 count:", (df['firmsize'] == 0).sum())
print("vetstat 0 count:", (df['vetstat'] == 0).sum())
print("diffany 0 count:", (df['diffany'] == 0).sum())
print("nativity 0 count:", (df['nativity'] == 0).sum())

pension 0 count: 0
firmsize 0 count: 0
vetstat 0 count: 0
diffany 0 count: 0
nativity 0 count: 0


In [15]:
# Step 4: Drop one of the two rows for household 8990 (person 10 or 12).
household_8990 = df[df['serial'] == 8990]
#Verify household 8990 has 3 rows not 4
print("Rows in household 8990:", len(household_8990))
print(household_8990[['serial', 'pernum','inctot','adjginc','fedtaxac']])
df = df.drop(index=13114)
household_8990_check = df[df['serial'] == 8990]
print("Row 12 in 8990 dropped, after drop:", len(household_8990_check))
print("New total rows:", len(df))

Rows in household 8990: 4
       serial  pernum  inctot  adjginc  fedtaxac
13103    8990       1   19600    19600     -3995
13105    8990       3   36000    36000      2438
13112    8990      10   60104    38351      1065
13114    8990      12   60104    38351      1065
Row 12 in 8990 dropped, after drop: 3
New total rows: 64695


# Steps 5 & 6 - Negative values and capped values

Negative values in fedtaxac, inincbus, incrent, and inctot are kept, they represent real losses and refundable credits like EITC. (Earned income tax credit).


Capped values (999, 999 in incrent/incdivid, age groups at 80/85) are kept but noted as limitations. Medians should be used for these values instead of means at the top of income distribution.


In [16]:
#Step 7: For any tax-rate analysis, drop or cap rows with very small AGI, and write down that rates below -50% were already removed from the file.
df = df[df['adjginc'] >= 1000]
#Verify
print("Rows remaining after AGI filter:", len(df))
print("Minimum AGI now:", min(df['adjginc']))

Rows remaining after AGI filter: 64335
Minimum AGI now: 1000


In [17]:
#Step 8: Rename yrimmig to years_in_us (or similar) and fix its data dictionary entry.
df = df.rename(columns={'yrimmig': 'years_in_us'})
# Column renamed to years_in_us (since moving to US) not actual year.
#Verify
print("'yrimmig' in columns:", 'yrimmig' in df.columns)
print("'years_in_us' in columns:", 'years_in_us' in df.columns)

'yrimmig' in columns: False
'years_in_us' in columns: True


In [19]:
#Step 9: Drop year, month, and asecflag. Tip: income year is 2023.
df = df.drop(columns=['year', 'month', 'asecflag'])
#Verify
print("Remaining columns:", len(df.columns))
print("'year' in columns:", 'year' in df.columns)
print("'month' in columns:", 'month' in df.columns)
print("'asecflag' in columns:",'asecflag' in df.columns)

Remaining columns: 56
'year' in columns: False
'month' in columns: False
'asecflag' in columns: False


# Survey Weights

All estimate model training should use asecwth survey weights. Use (asecwt, and spmwt for the SPM columns) in every estimate
The weighted average effective tax rate differs from the unweighted average.

(5.56% weighted) vs (5.17% unweighted).

ML teammate pass asecwth as sample_weight when training model.

In [24]:
#Step 11. Convert the column types to regular numpy types once cleaning is done.
#Used float for evertyhing since it can hanlde NaN values.
df = df.astype({col: 'float64' for col in df.select_dtypes(include=['Int64', 'Float64','int64','float64']).columns})

#Verify
print("Column type conversion:")
print(df.dtypes.value_counts())
print("\n Final Dataframe Shape:", df.shape)

Column type conversion:
float64    56
Name: count, dtype: int64

 Final Dataframe Shape: (64335, 56)
